In [ ]:
import json
import re
from pathlib import Path
from datetime import datetime, timezone
from io import StringIO

import pandas as pd
import requests

In [ ]:
FAMILY_MAP = {
    "Aire acondicionado": "AA",
    "Estacion de Carga Portatil": "GENKI",
    "TPMS": "TPMS",
    "TPMS Repuesto": "TPMS",
    "Climatizador": "CLIMATIZADOR",
    "Carjack": "CARJACK",
    "Arrancador": "CARJACK",
    "Inflador": "CARJACK",
    "Caldera": "CALDERA",
}

In [ ]:
def normalize_sku(x) -> str:
    return str(x).strip()


def split_images(s: str) -> list[str]:
    if not isinstance(s, str) or not s.strip():
        return []
    return [u.strip() for u in s.split(",") if u.strip()]


def to_bool(x):
    if pd.isna(x):
        return None
    return bool(x)

In [ ]:
def export_from_csv(csv_path_or_url: str, out_path: Path):
    if csv_path_or_url.startswith("http"):
        response = requests.get(csv_path_or_url)
        response.raise_for_status()
        df = pd.read_csv(StringIO(response.text))
    else:
        df = pd.read_csv(csv_path_or_url)

    items = []

    for _, r in df.iterrows():
        sku = normalize_sku(r["id"])
        title = str(r["title"]).strip() if pd.notna(r["title"]) else ""
        description = str(r["description"]).strip(
        ) if pd.notna(r["description"]) else ""
        availability = str(r["availability"]).strip().lower() == "in stock"
        condition = str(r["condition"]).strip().lower()
        price_ars = float(str(r["price"]).replace(",", "").replace(
            " ARS", "")) if pd.notna(r["price"]) else None
        sale_price_ars = float(str(r["sale_price"]).replace(",", "").replace(
            " ARS", "")) if pd.notna(r["sale_price"]) else None
        link = str(r["link"]).strip() if pd.notna(r["link"]) else None
        image_link = str(r["image_link"]).strip(
        ) if pd.notna(r["image_link"]) else None
        brand = str(r["brand"]).strip() if pd.notna(r["brand"]) else None
        google_product_category = str(r["google_product_category"]).strip(
        ) if pd.notna(r["google_product_category"]) else ""
        custom_label_1 = str(r["custom_label_1"]).strip(
        ) if pd.notna(r["custom_label_1"]) else ""

        # Determine family from google_product_category or title
        fam = "OTHER"
        for key in FAMILY_MAP:
            if key.lower() in google_product_category.lower() or key.lower() in title.lower():
                fam = FAMILY_MAP[key]
                break

        # Extract model from title (simple heuristic: first word after brand or something)
        model = title.split()[0] if title else ""

        items.append({
            "sku": sku,
            "family": fam,
            "family_raw": google_product_category,
            "model": model,
            "name": title,
            "price": {"ars": price_ars, "usd": None},
            "sale_price": {"ars": sale_price_ars, "usd": None},
            "availability": {
                "active": availability,
                "published": availability,
                "purchasable": availability,
            },
            "ids": {"product_id": sku},
            "taxonomy": {"categories": google_product_category, "custom_label_1": custom_label_1},
            "media": {"images": [image_link] if image_link else []},
            "links": {
                "product_page": link,
                "support_url": None,
                "sales_url": None,
                "whatsapp_support": None,
                "whatsapp_sales": None
            },
            "aliases": sorted(set(filter(None, [sku, model, title]))),
            "description": description,
        })

    catalog = {
        "schema_version": "catalog.v1",
        "generated_at": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
        "defaults": {"currency": "ARS", "country": "AR"},
        "source": {
            "type": "csv",
            "file": csv_path_or_url,
        },
        "items": sorted(items, key=lambda x: x["sku"]),
    }

    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(
        catalog, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Wrote {len(catalog['items'])} items to {out_path}")

In [ ]:
# Path or URL to CSV source
in_path = "https://neil.ar/facebook-feed.csv"
out_path = Path("../data/catalog.json")  # Output catalog.json path

if in_path.endswith(".csv") or in_path.startswith("http"):
    export_from_csv(in_path, out_path)
else:
    raise SystemExit("Unsupported input type. Use .csv or URL")

Wrote 71 items to ../data/catalog.json


In [6]:
# import camelot

# def export_from_pdf(pdf_path: Path, out_path: Path):
#     """
#     Minimal PDF support. Expects a table with columns like:
#     SKU | tipo de producto / Familia | Modelo | precio_ars

#     Requires: uv add camelot-py
#     Note: PDF table extraction can be brittle; Excel is strongly preferred.
#     """

#     tables = camelot.read_pdf(str(pdf_path), pages="all", flavor="stream")
#     if tables.n == 0:
#         raise SystemExit("No tables detected in PDF.")

#     # naive: take first table and rename columns
#     df = tables[0].df
#     df.columns = df.iloc[0]
#     df = df.iloc[1:].copy()

#     # try to match required columns
#     required = ["SKU", "tipo de producto / Familia", "Modelo", "precio_ars"]
#     missing = [c for c in required if c not in df.columns]
#     if missing:
#         raise SystemExit(f"PDF table missing columns: {missing}")

#     # build a minimal catalog (no enrichment)
#     items = []
#     for _, r in df.iterrows():
#         sku = normalize_sku(r["SKU"])
#         fam_raw = str(r["tipo de producto / Familia"]).strip()
#         fam = FAMILY_MAP.get(fam_raw, "OTHER")

#         # parse price
#         p = str(r["precio_ars"]).replace(".", "").replace(",", ".")
#         try:
#             price_ars = float(p)
#         except:
#             price_ars = None

#         items.append({
#             "sku": sku,
#             "family": fam,
#             "family_raw": fam_raw,
#             "model": str(r["Modelo"]).strip(),
#             "name": None,
#             "price": {"ars": price_ars, "usd": None},
#             "availability": {"active": None, "published": None, "purchasable": None},
#             "ids": {"product_id": None},
#             "taxonomy": {"categories": None},
#             "media": {"images": []},
#             "links": {"product_page": None, "support_url": None, "sales_url": None, "whatsapp_support": None, "whatsapp_sales": None},
#             "aliases": sorted(set(filter(None, [sku, str(r["Modelo"]).strip()]))),
#         })

#     catalog = {
#         "schema_version": "catalog.v1",
#         "generated_at": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
#         "defaults": {"currency": "ARS", "country": "AR"},
#         "source": {"type": "pdf", "file": pdf_path.name, "sheets": {}},
#         "items": sorted(items, key=lambda x: x["sku"]),
#     }

#     out_path.parent.mkdir(parents=True, exist_ok=True)
#     out_path.write_text(json.dumps(
#         catalog, ensure_ascii=False, indent=2), encoding="utf-8")
#     print(f"Wrote {len(catalog['items'])} items to {out_path}")